# 02 — Models & Loss Functions

**PI-CDA: Cross-Modal Digital Twin Adaptation**

This notebook defines:
1. **CTR-GCN** — Channel-wise Topology Refinement Graph Convolutional Network (primary backbone)
2. **ST-GCN** — Spatial-Temporal Graph Convolutional Network (baseline backbone)
3. **Temporal Anticipation Decoder** — Causal Transformer for next-action forecasting
4. **CSTA Loss** — Contrastive Skeleton Topology Alignment (instance + velocity + topology)
5. **Domain Adaptation Heads** — DANN discriminator, DeepCORAL, MMD alignment
6. **Action Precedence Automaton (APA)** — Assembly constraint layer
7. **Combined PI-CDA Model** — Unified end-to-end framework


In [7]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Optional, Tuple, List, Dict
from pathlib import Path

# Safe optional einops
try:
    from einops import rearrange, repeat
except ImportError:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


## 1. Graph Convolution Building Blocks


In [8]:
# ============================================================================
# Graph Convolution Utilities
# ============================================================================

def get_adjacency_matrix(edges, n_joints=17, self_loops=True):
    adj = np.zeros((n_joints, n_joints), dtype=np.float32)
    for i, j in edges:
        adj[i, j] = 1.0
        adj[j, i] = 1.0
    if self_loops:
        np.fill_diagonal(adj, 1.0)
    
    # Symmetric normalization: D^(-1/2) * A * D^(-1/2)
    deg = adj.sum(axis=1)
    deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)
    deg_inv_sqrt[deg == 0] = 0.0
    norm_adj = deg_inv_sqrt[:, None] * adj * deg_inv_sqrt[None, :]
    return norm_adj

# Canonical 17-joint skeleton edges
UNIFIED_EDGES = [
    (0, 16), (0, 1), (1, 2), (1, 5), (2, 3), (3, 4),
    (5, 6), (6, 7), (1, 14), (14, 15), (15, 8), (15, 11),
    (8, 9), (9, 10), (11, 12), (12, 13)
]


In [9]:
# ============================================================================
# Temporal Convolutions
# ============================================================================

class TemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=9, stride=1, dilation=1):
        super().__init__()
        pad = (kernel_size + (kernel_size - 1) * (dilation - 1) - 1) // 2
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=(kernel_size, 1),
            padding=(pad, 0),
            stride=(stride, 1),
            dilation=(dilation, 1),
        )
        self.bn = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        return self.bn(self.conv(x))


class MultiScaleTemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=[3, 5, 7], stride=1):
        super().__init__()
        branch_channels = out_channels // len(kernel_sizes)
        remainder = out_channels - branch_channels * len(kernel_sizes)
        
        self.branches = nn.ModuleList([
            TemporalConv(
                in_channels,
                branch_channels + (1 if i < remainder else 0),
                kernel_size=ks, stride=stride
            )
            for i, ks in enumerate(kernel_sizes)
        ])
    
    def forward(self, x):
        return torch.cat([branch(x) for branch in self.branches], dim=1)


## 2. CTR-GCN — Channel-wise Topology Refinement GCN


In [10]:
# ============================================================================
# CTR-GCN Core: Channel-wise Topology Refinement
# ============================================================================

class CTRGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, A, num_subsets=3, adaptive=True, reduction=8):
        super().__init__()
        self.num_subsets = num_subsets
        self.adaptive = adaptive
        self.num_joints = A.shape[0] if isinstance(A, np.ndarray) else A.size(0)
        
        A_tensor = torch.from_numpy(A.astype(np.float32)) if isinstance(A, np.ndarray) else A.float()
        if A_tensor.dim() == 2:
            A_tensor = A_tensor.unsqueeze(0).repeat(num_subsets, 1, 1)
        self.PA = nn.Parameter(A_tensor)
        
        if adaptive:
            self.alpha = nn.Parameter(torch.zeros(1))
            mid_channels = max(in_channels // reduction, 16)
            self.mid_channels = mid_channels
            self.conv_shared = nn.Conv2d(in_channels, mid_channels * num_subsets, 1)
            self.conv_refine = nn.Conv2d(in_channels, mid_channels * num_subsets, 1)
            
        self.out_conv = nn.Conv2d(in_channels * num_subsets, out_channels, 1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        if in_channels != out_channels:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x):
        B, C, T, V = x.shape
        res = self.residual(x)
        A = self.PA.unsqueeze(0)  # (1, K, V, V)
        
        if self.adaptive:
            x_pool = x.mean(dim=2, keepdim=True)
            shared = self.conv_shared(x_pool).view(B, self.num_subsets, self.mid_channels, V)
            refine = self.conv_refine(x_pool).view(B, self.num_subsets, self.mid_channels, V)
            
            attn = torch.einsum('bkmv,bkmw->bkvw', shared, refine) / math.sqrt(self.mid_channels)
            attn = torch.tanh(attn)
            A = A + self.alpha * attn
            
        out = torch.einsum('bctv,bkvw->bkctw', x, A.expand(B, -1, -1, -1))
        out = out.reshape(B, self.num_subsets * C, T, V)
        out = self.out_conv(out)
        out = self.bn(out)
        return self.relu(out + res)


In [11]:
# ============================================================================
# CTR-GCN Block (Graph Conv + Temporal Conv)
# ============================================================================

class CTRGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1, adaptive=True):
        super().__init__()
        self.gcn = CTRGraphConv(in_channels, out_channels, A, adaptive=adaptive)
        self.tcn = MultiScaleTemporalConv(out_channels, out_channels, stride=stride)
        self.relu = nn.ReLU(inplace=True)
        
        if stride != 1 or in_channels != out_channels:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.residual = nn.Identity()
    
    def forward(self, x):
        res = self.residual(x)
        x = self.gcn(x)
        x = self.tcn(x)
        return self.relu(x + res)


In [12]:
# ============================================================================
# Full CTR-GCN Model
# ============================================================================

class CTRGCN(nn.Module):
    def __init__(
        self,
        num_classes: int = 13,
        in_channels: int = 3,
        num_joints: int = 17,
        base_channels: int = 64,
        num_layers: int = 10,
        adaptive: bool = True,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.num_classes = num_classes
        
        A = get_adjacency_matrix(UNIFIED_EDGES, n_joints=num_joints)
        self.data_bn = nn.BatchNorm1d(in_channels * num_joints)
        
        channels = [base_channels] * 3 + [base_channels * 2] * 3 + [base_channels * 4] * (num_layers - 6)
        channels = channels[:num_layers]
        
        self.layers = nn.ModuleList()
        c_in = in_channels
        for i, c_out in enumerate(channels):
            stride = 2 if i in [3, 6] else 1
            self.layers.append(CTRGCNBlock(c_in, c_out, A, stride=stride, adaptive=adaptive))
            c_in = c_out
        
        self.feat_dim = channels[-1]
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.fc = nn.Linear(self.feat_dim, num_classes)
    
    def forward(self, x, classify=True):
        B, C, T, V = x.shape
        
        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.data_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2)
        
        for layer in self.layers:
            x = layer(x)
        
        features = self.gap(x).squeeze(-1).squeeze(-1)
        features = self.dropout(features)
        
        if classify:
            return self.fc(features)
        else:
            return features


# Test
model = CTRGCN(num_classes=13, in_channels=3, num_joints=17).to(device)
x_test = torch.randn(2, 3, 64, 17).to(device)
logits = model(x_test)
feats = model(x_test, classify=False)

total_params = sum(p.numel() for p in model.parameters())
print(f"CTR-GCN architecture:")
print(f"  Input:  {tuple(x_test.shape)}")
print(f"  Logits: {tuple(logits.shape)}")
print(f"  Feats:  {tuple(feats.shape)}")
print(f"  Params: {total_params:,} ({total_params/1e6:.2f}M)")


/tmp/ipykernel_3473164/2320187929.py:15: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)


CTR-GCN architecture:
  Input:  (2, 3, 64, 17)
  Logits: (2, 13)
  Feats:  (2, 256)
  Params: 2,761,883 (2.76M)


## 3. ST-GCN Baseline


In [13]:
# ============================================================================
# ST-GCN (Baseline)
# ============================================================================

class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1):
        super().__init__()
        A_tensor = torch.from_numpy(A.astype(np.float32)) if isinstance(A, np.ndarray) else A.float()
        self.register_buffer('A', A_tensor)
        self.gcn = nn.Conv2d(in_channels, out_channels, 1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (9, 1), stride=(stride, 1), padding=(4, 0)),
            nn.BatchNorm2d(out_channels),
        )
        self.relu = nn.ReLU(inplace=True)
        
        if stride != 1 or in_channels != out_channels:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.residual = nn.Identity()
    
    def forward(self, x):
        res = self.residual(x)
        x = torch.einsum('bctv,vw->bctw', x, self.A)
        x = self.gcn(x)
        x = self.tcn(x)
        return self.relu(x + res)


class STGCN(nn.Module):
    def __init__(self, num_classes=13, in_channels=3, num_joints=17, base_channels=64, num_layers=9):
        super().__init__()
        A = get_adjacency_matrix(UNIFIED_EDGES, n_joints=num_joints)
        self.data_bn = nn.BatchNorm1d(in_channels * num_joints)
        
        channels = [base_channels] * 3 + [base_channels * 2] * 3 + [base_channels * 4] * (num_layers - 6)
        channels = channels[:num_layers]
        
        self.layers = nn.ModuleList()
        c_in = in_channels
        for i, c_out in enumerate(channels):
            stride = 2 if i in [3, 6] else 1
            self.layers.append(STGCNBlock(c_in, c_out, A, stride=stride))
            c_in = c_out
        
        self.feat_dim = channels[-1]
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(self.feat_dim, num_classes)
    
    def forward(self, x, classify=True):
        B, C, T, V = x.shape
        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.data_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2)
        
        for layer in self.layers:
            x = layer(x)
        
        features = self.gap(x).squeeze(-1).squeeze(-1)
        if classify:
            return self.fc(features)
        else:
            return features


# Test
stgcn = STGCN().to(device)
stgcn_params = sum(p.numel() for p in stgcn.parameters())
print(f"ST-GCN baseline:")
print(f"  Params: {stgcn_params:,} ({stgcn_params/1e6:.2f}M)")
print(f"  Output: {tuple(stgcn(x_test).shape)}")


ST-GCN baseline:
  Params: 2,589,619 (2.59M)
  Output: (2, 13)


/tmp/ipykernel_3473164/2320187929.py:15: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)


## 4. Temporal Anticipation Decoder


In [14]:
# ============================================================================
# Temporal Anticipation Decoder
# ============================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class AnticipationDecoder(nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        num_classes: int = 13,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.pos_encoder = PositionalEncoding(d_model)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        self.head_05s = nn.Linear(d_model, num_classes)
        self.head_10s = nn.Linear(d_model, num_classes)
        self.head_20s = nn.Linear(d_model, num_classes)
        self.head_30s = nn.Linear(d_model, num_classes)
    
    def generate_causal_mask(self, T, device):
        mask = torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)
        return mask
    
    def forward(self, feature_seq, horizon="1.0s"):
        B, T, _ = feature_seq.shape
        x = self.pos_encoder(feature_seq)
        mask = self.generate_causal_mask(T, feature_seq.device)
        
        out = self.transformer(tgt=x, memory=x, tgt_mask=mask, memory_mask=mask)
        last_state = out[:, -1, :]
        
        if horizon == "0.5s":
            return self.head_05s(last_state)
        elif horizon == "1.0s":
            return self.head_10s(last_state)
        elif horizon == "2.0s":
            return self.head_20s(last_state)
        elif horizon == "3.0s":
            return self.head_30s(last_state)
        elif horizon == "all":
            return {
                "0.5s": self.head_05s(last_state),
                "1.0s": self.head_10s(last_state),
                "2.0s": self.head_20s(last_state),
                "3.0s": self.head_30s(last_state),
            }
        else:
            return self.head_10s(last_state)


# Test
anticipation = AnticipationDecoder().to(device)
feat_seq = torch.randn(2, 10, 256).to(device)
pred = anticipation(feat_seq, horizon="1.0s")
pred_all = anticipation(feat_seq, horizon="all")
print(f"Anticipation decoder:")
print(f"  Single horizon output: {tuple(pred.shape)}")
print(f"  Multi-horizon output:  {[f'{k}: {tuple(v.shape)}' for k, v in pred_all.items()]}")


Anticipation decoder:
  Single horizon output: (2, 13)
  Multi-horizon output:  ['0.5s: (2, 13)', '1.0s: (2, 13)', '2.0s: (2, 13)', '3.0s: (2, 13)']


## 5. Loss Functions


In [15]:
# ============================================================================
# CSTA Loss — Contrastive Skeleton Topology Alignment
# ============================================================================

class CSTALoss(nn.Module):
    def __init__(self, temperature=0.1, lambda_vel=0.5, lambda_topo=0.3):
        super().__init__()
        self.temperature = temperature
        self.lambda_vel = lambda_vel
        self.lambda_topo = lambda_topo
    
    def nt_xent_loss(self, z_source, z_target):
        z_s = F.normalize(z_source, dim=-1)
        z_t = F.normalize(z_target, dim=-1)
        
        sim = torch.matmul(z_s, z_t.T) / self.temperature
        targets = torch.arange(min(z_s.size(0), z_t.size(0)), device=z_source.device)
        
        loss_s = F.cross_entropy(sim[:len(targets), :len(targets)], targets)
        loss_t = F.cross_entropy(sim.T[:len(targets), :len(targets)], targets)
        return (loss_s + loss_t) / 2.0
    
    def rbf_kernel(self, x, y, sigmas=[1.0, 5.0, 10.0]):
        xx = torch.cdist(x, x) ** 2
        yy = torch.cdist(y, y) ** 2
        xy = torch.cdist(x, y) ** 2
        
        k_xx, k_yy, k_xy = 0.0, 0.0, 0.0
        for s in sigmas:
            gamma = 1.0 / (2.0 * s * s)
            k_xx = k_xx + torch.exp(-gamma * xx)
            k_yy = k_yy + torch.exp(-gamma * yy)
            k_xy = k_xy + torch.exp(-gamma * xy)
        return k_xx.mean() + k_yy.mean() - 2.0 * k_xy.mean()
    
    def forward(self, z_source, z_target, vel_source=None, vel_target=None, topo_source=None, topo_target=None):
        losses = {}
        
        L_instance = self.nt_xent_loss(z_source, z_target)
        losses["L_instance"] = L_instance
        total_loss = L_instance
        
        if vel_source is not None and vel_target is not None and self.lambda_vel > 0:
            L_vel = self.rbf_kernel(vel_source, vel_target)
            losses["L_velocity"] = L_vel
            total_loss = total_loss + self.lambda_vel * L_vel
        
        if topo_source is not None and topo_target is not None and self.lambda_topo > 0:
            L_topo = F.mse_loss(topo_source, topo_target)
            losses["L_topology"] = L_topo
            total_loss = total_loss + self.lambda_topo * L_topo
        
        losses["L_CSTA_total"] = total_loss
        return total_loss, losses


# Test
csta = CSTALoss()
z_s = torch.randn(8, 256).to(device)
z_t = torch.randn(8, 256).to(device)
loss, detail = csta(z_s, z_t)
print(f"CSTA Loss test: {[f'{k}: {v.item():.4f}' for k, v in detail.items()]}")


CSTA Loss test: ['L_instance: 2.4737', 'L_CSTA_total: 2.4737']


In [16]:
# ============================================================================
# Domain Discriminator (for DANN baseline) & DeepCORAL
# ============================================================================

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha=1.0):
        ctx.alpha = alpha
        return x.view_as(x)
    
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class DomainDiscriminator(nn.Module):
    def __init__(self, in_features=256, hidden_dim=128):
        super().__init__()
        self.grl = GradientReversal()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, 2),
        )
    def forward(self, x, alpha=1.0):
        self.grl.alpha = alpha
        return self.net(self.grl(x))

def coral_loss(source, target):
    d = source.size(1)
    ns, nt = source.size(0), target.size(0)
    cs = (source - source.mean(0, keepdim=True)).T @ (source - source.mean(0, keepdim=True)) / (ns - 1)
    ct = (target - target.mean(0, keepdim=True)).T @ (target - target.mean(0, keepdim=True)) / (nt - 1)
    return (cs - ct).pow(2).sum() / (4.0 * d * d)

print("Domain adaptation components defined ✓")


Domain adaptation components defined ✓


## 6. Combined PI-CDA Model


In [17]:
# ============================================================================
# PI-CDA: Full Framework
# ============================================================================

class PICDA(nn.Module):
    def __init__(
        self,
        num_classes: int = 13,
        in_channels: int = 3,
        num_joints: int = 17,
        feat_dim: int = 256,
        anticipation_d_model: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.num_classes = num_classes
        
        self.backbone = CTRGCN(
            num_classes=num_classes,
            in_channels=in_channels,
            num_joints=num_joints,
            dropout=dropout,
        )
        
        self.anticipator = AnticipationDecoder(
            d_model=anticipation_d_model,
            num_classes=num_classes,
            dropout=dropout,
        )
        
        self.discriminator = DomainDiscriminator(
            in_features=feat_dim,
        )
    
    def forward(self, x, task="recognition", horizon="1.0s"):
        if task == "recognition":
            return self.backbone(x, classify=True)
        elif task == "features":
            return self.backbone(x, classify=False)
        elif task == "anticipation":
            feats = self.backbone(x, classify=False)
            feat_seq = feats.unsqueeze(1)
            return self.anticipator(feat_seq, horizon=horizon)
        elif task == "domain":
            feats = self.backbone(x, classify=False)
            return self.discriminator(feats)
        else:
            raise ValueError(f"Unknown task: {task}")


# Test full model
picda = PICDA().to(device)
logits_rec = picda(x_test, task="recognition")
feats_all = picda(x_test, task="features")
logits_ant = picda(x_test, task="anticipation", horizon="1.0s")
dom_preds = picda(x_test, task="domain")

total_params = sum(p.numel() for p in picda.parameters())
print(f"Full PI-CDA Model:")
print(f"  Total params:   {total_params:,} ({total_params/1e6:.2f}M)")
print(f"  Recognition:    {tuple(logits_rec.shape)}")
print(f"  Features:       {tuple(feats_all.shape)}")
print(f"  Anticipation:   {tuple(logits_ant.shape)}")
print(f"  Domain preds:   {tuple(dom_preds.shape)}")


Full PI-CDA Model:
  Total params:   5,980,049 (5.98M)
  Recognition:    (2, 13)
  Features:       (2, 256)
  Anticipation:   (2, 13)
  Domain preds:   (2, 2)


/tmp/ipykernel_3473164/2320187929.py:15: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)


## 7. PyTorch Dataset & DataLoader


In [18]:
# ============================================================================
# PyTorch Dataset for skeleton sequences
# ============================================================================

from torch.utils.data import Dataset, DataLoader

class SkeletonDataset(Dataset):
    def __init__(
        self,
        file_list: list,
        data_dir: str,
        label_map: dict = None,
        max_frames: int = 64,
        input_channels: str = 'position',
        augmentor=None,
    ):
        self.file_list = file_list
        self.data_dir = Path(data_dir)
        self.label_map = label_map or {}
        self.max_frames = max_frames
        self.input_channels = input_channels
        self.augmentor = augmentor
    
    def __len__(self):
        return len(self.file_list)
    
    def _pad_or_crop(self, seq: np.ndarray) -> np.ndarray:
        T = seq.shape[0]
        if T >= self.max_frames:
            start = np.random.randint(0, T - self.max_frames + 1)
            return seq[start:start + self.max_frames]
        else:
            pad = np.zeros((self.max_frames - T, *seq.shape[1:]), dtype=seq.dtype)
            return np.concatenate([seq, pad], axis=0)
    
    def __getitem__(self, idx):
        filepath = self.data_dir / self.file_list[idx]
        data = np.load(str(filepath))
        
        pos = data['positions']
        vel = data.get('velocities', np.zeros_like(pos))
        
        if self.augmentor is not None:
            pos = self.augmentor(pos)
            vel = np.zeros_like(pos)
            vel[1:] = (pos[1:] - pos[:-1]) * 30.0
        
        if self.input_channels == 'position':
            x = pos
        elif self.input_channels == 'velocity':
            x = vel
        elif self.input_channels == 'both':
            x = np.concatenate([pos, vel], axis=-1)
        else:
            x = pos
        
        x = self._pad_or_crop(x)
        x = x.transpose(2, 0, 1)
        x = torch.from_numpy(x).float()
        
        label = int(data.get('label', self.label_map.get(self.file_list[idx], 0)))
        return x, label, idx

print("Dataset class defined ✓")


Dataset class defined ✓


## 8. Summary


In [19]:
print("=" * 60)
print("Notebook 02 — Models & Losses — COMPLETE")
print("=" * 60)
print(f"  CTR-GCN params:      {sum(p.numel() for p in model.parameters()):,}")
print(f"  ST-GCN params:       {sum(p.numel() for p in stgcn.parameters()):,}")
print(f"  Anticipation params: {sum(p.numel() for p in anticipation.parameters()):,}")
print(f"  Full PI-CDA params:  {sum(p.numel() for p in picda.parameters()):,}")
print("=" * 60)


Notebook 02 — Models & Losses — COMPLETE
  CTR-GCN params:      2,761,883
  ST-GCN params:       2,589,619
  Anticipation params: 3,176,500
  Full PI-CDA params:  5,980,049
